# 1. Bronze Layer

In [1]:
from pyspark.sql.functions import col, explode, split, when, lit

# 1. Read Raw Data (Removed multiline to enable JSON Lines parsing)
bronze_path = "Files/raw/bwt/*/*/*/*.json"
raw_df = spark.read.json(bronze_path)

display(raw_df)

print(f"Total raw records loaded: {raw_df.count()}")

StatementMeta(, 156bc72f-6d44-45cf-952d-5b04ea727823, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82285f86-32c3-433c-8987-28c0f76dd23e)

Total raw records loaded: 17598


In [ ]:

raw_df.printSchema()

StatementMeta(, 052e128c-b9c4-4386-9441-6ce68630d3eb, 4, Finished, Available, Finished, False)

root
 |-- border: string (nullable = true)
 |-- commercial_vehicle_lanes: struct (nullable = true)
 |    |-- FAST_lanes: struct (nullable = true)
 |    |    |-- delay_minutes: string (nullable = true)
 |    |    |-- lanes_open: string (nullable = true)
 |    |    |-- operational_status: string (nullable = true)
 |    |    |-- update_time: string (nullable = true)
 |    |-- maximum_lanes: string (nullable = true)
 |    |-- standard_lanes: struct (nullable = true)
 |    |    |-- delay_minutes: string (nullable = true)
 |    |    |-- lanes_open: string (nullable = true)
 |    |    |-- operational_status: string (nullable = true)
 |    |    |-- update_time: string (nullable = true)
 |-- construction_notice: string (nullable = true)
 |-- crossing_name: string (nullable = true)
 |-- date: string (nullable = true)
 |-- hours: string (nullable = true)
 |-- passenger_vehicle_lanes: struct (nullable = true)
 |    |-- NEXUS_SENTRI_lanes: struct (nullable = true)
 |    |    |-- delay_minutes: stri

# 2. Silver Layer 

In [2]:
from pyspark.sql.functions import col, lit, when, input_file_name, regexp_extract, to_timestamp, concat_ws

# 1. Read Raw Data (MULTILINE REMOVED)
bronze_path = "Files/raw/bwt/*/*/*/*.json"
raw_df = spark.read.json(bronze_path)

# 2. Extract UTC Time from the Pipeline's Folder Structure
df_with_time = raw_df.withColumn("file_path", input_file_name()) \
   .withColumn("utc_year", regexp_extract(col("file_path"), r"(\d{4})/(\d{2})/(\d{2})/(\d{2})\.json", 1)) \
   .withColumn("utc_month", regexp_extract(col("file_path"), r"(\d{4})/(\d{2})/(\d{2})/(\d{2})\.json", 2)) \
   .withColumn("utc_day", regexp_extract(col("file_path"), r"(\d{4})/(\d{2})/(\d{2})/(\d{2})\.json", 3)) \
   .withColumn("utc_hour", regexp_extract(col("file_path"), r"(\d{4})/(\d{2})/(\d{2})/(\d{2})\.json", 4)) \
   .withColumn("snapshot_utc", to_timestamp(concat_ws("-", col("utc_year"), col("utc_month"), col("utc_day"), col("utc_hour")), "yyyy-MM-dd-HH"))

# 3. Define a helper function to unnest specific lane types
def extract_lane(df, pillar, lane):
    return df.select(
       col("port_number"),
       col("port_name"),
       col("border"),
       col("snapshot_utc"),
       lit(pillar).alias("traffic_pillar"),
       lit(lane).alias("lane_type"),
       col(f"{pillar}.{lane}.delay_minutes").alias("delay_minutes"),
       col(f"{pillar}.{lane}.lanes_open").alias("lanes_open"),
       col(f"{pillar}.{lane}.operational_status").alias("operational_status")
   ).filter(col("operational_status").isNotNull() & (col("operational_status") != "N/A"))

# 4. Unpivot the massive nested JSON into a clean Fact table structure
df_comm_std = extract_lane(df_with_time, "commercial_vehicle_lanes", "standard_lanes")
df_comm_fast = extract_lane(df_with_time, "commercial_vehicle_lanes", "FAST_lanes")
df_pass_std = extract_lane(df_with_time, "passenger_vehicle_lanes", "standard_lanes")
df_pass_nex = extract_lane(df_with_time, "passenger_vehicle_lanes", "NEXUS_SENTRI_lanes")
df_pass_ready = extract_lane(df_with_time, "passenger_vehicle_lanes", "ready_lanes")
df_ped_std = extract_lane(df_with_time, "pedestrian_lanes", "standard_lanes")
df_ped_ready = extract_lane(df_with_time, "pedestrian_lanes", "ready_lanes")

# Combine all the lane types into a single vertical dataframe
unpivoted_df = df_comm_std.unionByName(df_comm_fast) \
   .unionByName(df_pass_std) \
   .unionByName(df_pass_nex) \
   .unionByName(df_pass_ready) \
   .unionByName(df_ped_std) \
   .unionByName(df_ped_ready)

# 5. Apply the Data Integrity Rules
silver_df = unpivoted_df.withColumn(
   "wait_time_min",
   when(col("operational_status").isin("Lanes Closed", "Update Pending"), lit(None).cast("int"))
   .otherwise(col("delay_minutes").cast("int"))
).withColumn(
   "lanes_open", col("lanes_open").cast("int")
).drop("delay_minutes")

# 6. Save as a Delta Table to your Lakehouse
silver_df.write.format("delta").mode("overwrite").saveAsTable("Fact_WaitTimes_Silver_New")

print(f"Total processed Silver records: {silver_df.count()}")

display(silver_df)

StatementMeta(, 156bc72f-6d44-45cf-952d-5b04ea727823, 4, Finished, Available, Finished, False)

Total processed Silver records: 78619


SynapseWidget(Synapse.DataFrame, 02c7edf2-8295-480f-b59f-6740c9ff5f1a)

In [3]:
print(silver_df.count())

StatementMeta(, 052e128c-b9c4-4386-9441-6ce68630d3eb, 6, Finished, Available, Finished, False)

77264


In [5]:
df_with_time.select("utc_year", "utc_month", "utc_day", "utc_hour") \
    .distinct() \
    .orderBy("utc_year", "utc_month", "utc_day", "utc_hour") \
    .show(50)

StatementMeta(, 5095a4db-b89b-4f85-b09a-a5e6fb368637, 7, Finished, Available, Finished, False)

+--------+---------+-------+--------+
|utc_year|utc_month|utc_day|utc_hour|
+--------+---------+-------+--------+
|    2026|       07|     23|      18|
|    2026|       07|     23|      19|
|    2026|       07|     23|      20|
|    2026|       07|     23|      21|
|    2026|       07|     23|      22|
|    2026|       07|     23|      23|
|    2026|       07|     24|      00|
|    2026|       07|     24|      01|
|    2026|       07|     24|      02|
|    2026|       07|     24|      03|
|    2026|       07|     24|      04|
|    2026|       07|     24|      05|
|    2026|       07|     24|      06|
|    2026|       07|     24|      07|
|    2026|       07|     24|      08|
|    2026|       07|     24|      09|
|    2026|       07|     24|      10|
|    2026|       07|     24|      11|
|    2026|       07|     24|      12|
|    2026|       07|     24|      13|
|    2026|       07|     24|      14|
|    2026|       07|     24|      15|
|    2026|       07|     24|      16|
|    2026|  

In [5]:
files = spark.read.format("binaryFile").load(bronze_path)
print(files.count())
files.select("path").show(50, truncate=False)

StatementMeta(, 052e128c-b9c4-4386-9441-6ce68630d3eb, 7, Finished, Available, Finished, False)

205
+---------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                               |
+---------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://2383d835-68ef-4b30-aff3-37cf3d1ba74d@onelake.dfs.fabric.microsoft.com/8dfbd124-80d9-477d-8d1f-a4c9c3089898/Files/raw/bwt/2026/07/27/15.json|
|abfss://2383d835-68ef-4b30-aff3-37cf3d1ba74d@onelake.dfs.fabric.microsoft.com/8dfbd124-80d9-477d-8d1f-a4c9c3089898/Files/raw/bwt/2026/07/27/17.json|
|abfss://2383d835-68ef-4b30-aff3-37cf3d1ba74d@onelake.dfs.fabric.microsoft.com/8dfbd124-80d9-477d-8d1f-a4c9c3089898/Files/raw/bwt/2026/07/27/16.json|
|abfss://2383d835-68ef-4b30-aff3-37cf3d1ba74d@onelake.dfs.fabric.microsoft.com/8dfbd124-80d9-477

In [5]:
print(raw_df.count())

StatementMeta(, 052e128c-b9c4-4386-9441-6ce68630d3eb, 9, Finished, Available, Finished, False)

17343


In [4]:
df_with_time.select("snapshot_utc").distinct().orderBy("snapshot_utc").show(50, truncate=False)

StatementMeta(, 85594dc6-1c24-4e12-98c4-492f28497b9d, 6, Finished, Available, Finished, False)

+-------------------+
|snapshot_utc       |
+-------------------+
|2026-07-23 18:00:00|
|2026-07-23 19:00:00|
|2026-07-23 20:00:00|
|2026-07-23 21:00:00|
|2026-07-23 22:00:00|
|2026-07-23 23:00:00|
|2026-07-24 00:00:00|
|2026-07-24 01:00:00|
|2026-07-24 02:00:00|
|2026-07-24 03:00:00|
|2026-07-24 04:00:00|
|2026-07-24 05:00:00|
|2026-07-24 06:00:00|
|2026-07-24 07:00:00|
|2026-07-24 08:00:00|
|2026-07-24 09:00:00|
|2026-07-24 10:00:00|
|2026-07-24 11:00:00|
|2026-07-24 12:00:00|
|2026-07-24 13:00:00|
|2026-07-24 14:00:00|
|2026-07-24 15:00:00|
|2026-07-24 16:00:00|
|2026-07-24 17:00:00|
|2026-07-24 18:00:00|
|2026-07-24 19:00:00|
|2026-07-24 20:00:00|
|2026-07-24 21:00:00|
|2026-07-24 22:00:00|
|2026-07-24 23:00:00|
|2026-07-25 00:00:00|
|2026-07-25 01:00:00|
|2026-07-25 02:00:00|
|2026-07-25 03:00:00|
|2026-07-25 04:00:00|
|2026-07-25 05:00:00|
|2026-07-25 06:00:00|
|2026-07-25 07:00:00|
|2026-07-25 08:00:00|
|2026-07-25 09:00:00|
|2026-07-25 10:00:00|
|2026-07-25 11:00:00|
|2026-07-2

In [3]:
from pyspark.sql.functions import col, input_file_name

# 1. Read with recursive lookup (bypassing wildcard bugs) and strict error tracking
diagnostic_path = "Files/raw/bwt/"
diag_df = spark.read.option("multiline", "true") \
    .option("recursiveFileLookup", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json(diagnostic_path)

# 2. Count raw array elements (Should be roughly 43 files * 118 ports = ~5,000)
total_raw = diag_df.count()
print(f"Total raw JSON elements read: {total_raw}")

# 3. Check for API failures / corrupt files
if "_corrupt_record" in diag_df.columns:
    corrupt_count = diag_df.filter(col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt records found (API errors/HTML pages): {corrupt_count}")
else:
    print("No corrupt records column generated.")

# 4. Count distinct files successfully processed
file_count = diag_df.select(input_file_name()).distinct().count()
print(f"Total distinct files successfully parsed by Spark: {file_count}")

# 5. Display a sample of the file paths Spark actually sees
display(diag_df.select(input_file_name()).distinct().limit(10))

StatementMeta(, 156bc72f-6d44-45cf-952d-5b04ea727823, 5, Finished, Available, Finished, False)

Total raw JSON elements read: 208
No corrupt records column generated.
Total distinct files successfully parsed by Spark: 208


SynapseWidget(Synapse.DataFrame, 4fcee702-163a-45b6-a948-c61ca6e2fabe)

# 3. Gold Layer

In [4]:
from pyspark.sql.functions import col, concat_ws, to_date, hour, lit

# 1. Engineer Surrogate Keys and Split Time
gold_prep_df = silver_df.withColumn(
    "lane_id", concat_ws("_", col("traffic_pillar"), col("lane_type"))
).withColumn(
    "snapshot_date", to_date(col("snapshot_utc"))
).withColumn(
    "snapshot_hour", hour(col("snapshot_utc"))
)

# 2. Repair Dim_Port
dim_port = gold_prep_df.select("port_number", "port_name", "border").distinct()
dim_port.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Dim_Port")

# 3. Repair Dim_Lane
dim_lane = gold_prep_df.select("lane_id", "traffic_pillar", "lane_type").distinct()
dim_lane.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Dim_Lane_New")

# 4. Create Dim_Date & Dim_Time (Required for Direct Lake DAX)
dim_date = gold_prep_df.select(col("snapshot_date").alias("date")).distinct()
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Dim_Date")

dim_time = gold_prep_df.select(col("snapshot_hour").alias("hour_of_day")).distinct()
dim_time.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Dim_Time")

# 5. Repair Fact_WaitTimes_Gold
fact_gold = gold_prep_df.select(
    "port_number",
    "lane_id",
    "snapshot_date",
    "snapshot_hour",
    "lanes_open",
    "operational_status",
    "wait_time_min"
)
fact_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Fact_WaitTimes_Gold_New")

print("Enterprise Star Schema Patched and Ready for BI/ML!")

StatementMeta(, 156bc72f-6d44-45cf-952d-5b04ea727823, 6, Finished, Available, Finished, True)

Enterprise Star Schema Patched and Ready for BI/ML!
